# CyberSafe USeP — Thematic & Content Analysis
### Pure Python · No API · Runs 100% Offline

**Place these files in the same folder as this notebook before running:**
- `cybersafe_data.xlsx`
- `Minutes_1_.docx`
- `results3.xlsx`

Run cells **top to bottom**. Every step saves its output as a `.csv` or `.txt` file.

---
## ⚙️ Setup — Install & Import

In [1]:
!pip install openpyxl python-docx pandas


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import re
import warnings
from pathlib import Path
from collections import defaultdict

import pandas as pd
from docx import Document

warnings.filterwarnings('ignore')

# ── Helpers ───────────────────────────────────────────────────────────────────
SKIP = {'n/a', 'na', 'none', 'nan', '', '-', '.', 'n.a.'}

def is_blank(val):
    return str(val).strip().lower() in SKIP

def clean(val):
    return str(val).strip()

def save_txt(path, text):
    Path(path).write_text(text, encoding='utf-8')
    print(f'  Saved -> {path}')

def count_term(term, text):
    """Case-insensitive whole-word frequency count."""
    pattern = r'\b' + re.escape(term.lower()) + r'\b'
    return len(re.findall(pattern, text.lower()))

print('Setup complete.')

Setup complete.


---
## Step 0 — Load All Three Files

In [9]:
# ── cybersafe_data.xlsx ───────────────────────────────────────────────────────
df = pd.read_excel('data/cybersafe_data.xlsx', header=0)
print(f'Survey rows: {len(df)}  |  Columns: {len(df.columns)}')

# Column references — 0-indexed (column number minus 1)
COL_ROLE = df.columns[2]   # Column 3  — Role
COL_DEPT = df.columns[3]   # Column 4  — Department
COL_Q7   = df.columns[6]   # Column 7
COL_Q8   = df.columns[7]   # Column 8
COL_Q33  = df.columns[32]  # Column 33
COL_Q34  = df.columns[33]  # Column 34
COL_Q50  = df.columns[49]  # Column 50
COL_Q53  = df.columns[52]  # Column 53

OPEN_COLS = {
    'Q7  (Data privacy policy)':      COL_Q7,
    'Q8  (Cybersecurity safeguards)':  COL_Q8,
    'Q33 (Worries - data privacy)':    COL_Q33,
    'Q34 (Worries - cybersecurity)':   COL_Q34,
    'Q50 (Additional tools/support)':  COL_Q50,
    'Q53 (Additional comments)':       COL_Q53,
}

print('\nColumn mapping:')
for label, col in OPEN_COLS.items():
    print(f'  {label} -> "{col}"')


Survey rows: 600  |  Columns: 53

Column mapping:
  Q7  (Data privacy policy) -> "Data Privacy 

Protecting sensitive and personal data from unwanted access, use, or disclosure is known as data privacy. In a university context, this entails protecting administrative records, research data, faculty information, and student records. Maintaining data privacy adheres to legal requirements as stated in RA 10173 or the Data Privacy Act of 2012 in the Philippines. It safeguards people's identities and personal information, and supports academic integrity. It is essential for protecting both digital and physical information systems and building confidence with the universe community.

To what extent do you think our university's current data privacy policies adequately protect our personal data?"
  Q8  (Cybersecurity safeguards) -> "Cybersecurity

The term "cybersecurity" describes the procedures, tools, and methods used to defend digital networks, systems, and data against online dangers such

In [10]:
# ── Minutes_1_.docx ───────────────────────────────────────────────────────────
doc = Document('data/Minutes(1).docx')
INTERVIEW_PARAS = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
INTERVIEW_TEXT  = '\n'.join(INTERVIEW_PARAS)
save_txt('interview_transcript.txt', INTERVIEW_TEXT)
print(f'Interview paragraphs loaded: {len(INTERVIEW_PARAS)}')

# ── results3.xlsx ─────────────────────────────────────────────────────────────
df_nvivo = pd.read_excel('data/results3.xlsx', header=0)
print(f'\nresults3.xlsx shape: {df_nvivo.shape}')
print(f'Columns: {list(df_nvivo.columns)}')

quote_candidates = [c for c in df_nvivo.columns
                    if any(k in str(c).lower() for k in ['quote', 'response', 'text', 'verbatim', 'content'])]
QUOTE_COL = quote_candidates[0] if quote_candidates else df_nvivo.columns[-1]
print(f'Using quotes column: "{QUOTE_COL}"')

NVIVO_TEXT = ' '.join(df_nvivo[QUOTE_COL].dropna().astype(str).tolist())
save_txt('nvivo_quotes.txt', NVIVO_TEXT)
print(f'NVivo quotes loaded: {len(df_nvivo)} rows')

  Saved -> interview_transcript.txt
Interview paragraphs loaded: 219

results3.xlsx shape: (16, 4)
Columns: ['Clustered Themes', 'Emerging Themes', 'Remarks (Direct Quotes from Interviewees)', 'Participants']
Using quotes column: "Remarks (Direct Quotes from Interviewees)"
  Saved -> nvivo_quotes.txt
NVivo quotes loaded: 16 rows


---
## 🧹 Data Cleaning & Missing Data Handling
### Run this BEFORE any analysis. It audits and cleans all three datasets.

In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOCK 1 — SURVEY (cybersafe_data.xlsx)
# ══════════════════════════════════════════════════════════════════════════════
print('=' * 60)
print('SURVEY DATA — PRE-CLEANING AUDIT')
print('=' * 60)

total_rows = len(df)
print(f'Total rows loaded          : {total_rows}')

# 1a. Fully empty rows (all columns NaN)
fully_empty = df.isnull().all(axis=1).sum()
print(f'Fully empty rows           : {fully_empty}')

# 1b. Missing values per key column
key_cols = [COL_ROLE, COL_DEPT, COL_Q7, COL_Q8, COL_Q33, COL_Q34, COL_Q50, COL_Q53]
print('\nMissing / null counts per key column:')
for col in key_cols:
    n_null  = df[col].isnull().sum()
    n_blank = df[col].astype(str).str.strip().str.lower().isin(SKIP).sum()
    print(f'  {str(col):<45} null={n_null:>3}  placeholder={n_blank:>3}')

# 1c. Duplicate rows
n_dupes = df.duplicated().sum()
print(f'\nFully duplicate rows       : {n_dupes}')

SURVEY DATA — PRE-CLEANING AUDIT
Total rows loaded          : 600
Fully empty rows           : 0

Missing / null counts per key column:
  Directions
Please complete your profile below and indicate your level of agreement on the indicators pertaining to cybersecurity and Data Privacy. null=  0  placeholder=  0
  Department/Unit                               null=  0  placeholder=  0
  Data Privacy 

Protecting sensitive and personal data from unwanted access, use, or disclosure is known as data privacy. In a university context, this entails protecting administrative records, research data, faculty information, and student records. Maintaining data privacy adheres to legal requirements as stated in RA 10173 or the Data Privacy Act of 2012 in the Philippines. It safeguards people's identities and personal information, and supports academic integrity. It is essential for protecting both digital and physical information systems and building confidence with the universe community.

To what e

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOCK 2 — CLEAN THE SURVEY
# ══════════════════════════════════════════════════════════════════════════════

df_clean = df.copy()

# Step 1 — Drop fully empty rows
before = len(df_clean)
df_clean = df_clean.dropna(how='all')
print(f'Dropped fully empty rows   : {before - len(df_clean)}')

# Step 2 — Drop exact duplicate rows
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f'Dropped duplicate rows     : {before - len(df_clean)}')

# Step 3 — Standardise Role and Department: strip whitespace, title-case
df_clean[COL_ROLE] = df_clean[COL_ROLE].astype(str).str.strip().str.title()
df_clean[COL_DEPT] = df_clean[COL_DEPT].astype(str).str.strip().str.title()

# Step 4 — Fill missing Role/Dept with 'Unknown' so no row is silently dropped
unknown_role = df_clean[COL_ROLE].isin(['Nan', 'None', '', 'N/A']).sum()
unknown_dept = df_clean[COL_DEPT].isin(['Nan', 'None', '', 'N/A']).sum()
df_clean[COL_ROLE] = df_clean[COL_ROLE].replace({'Nan': 'Unknown', 'None': 'Unknown',
                                                   '': 'Unknown', 'N/A': 'Unknown'})
df_clean[COL_DEPT] = df_clean[COL_DEPT].replace({'Nan': 'Unknown', 'None': 'Unknown',
                                                   '': 'Unknown', 'N/A': 'Unknown'})
print(f'Role filled as Unknown     : {unknown_role}')
print(f'Dept filled as Unknown     : {unknown_dept}')

# Step 5 — Normalise open-ended columns:
#   - Strip leading/trailing whitespace
#   - Collapse multiple internal spaces
#   - Replace placeholder strings with actual NaN
PLACEHOLDER_PATTERN = re.compile(
    r'^\s*(n/?a\.?|none|no|nope|nothing|wala|\-|\.|\.\.\.+|no comment|no answer)\s*$',
    re.IGNORECASE
)

cleaned_cells = 0
for col in [COL_Q7, COL_Q8, COL_Q33, COL_Q34, COL_Q50, COL_Q53]:
    original = df_clean[col].astype(str)
    # Strip and collapse spaces
    df_clean[col] = original.str.strip().str.replace(r'\s+', ' ', regex=True)
    # Replace placeholders with NaN
    mask = df_clean[col].str.match(PLACEHOLDER_PATTERN, na=False)
    cleaned_cells += mask.sum()
    df_clean.loc[mask, col] = float('nan')

print(f'Placeholder cells -> NaN   : {cleaned_cells}')

# Step 6 — Rows where ALL open-ended columns are NaN (no text contribution at all)
all_oe_null = df_clean[[COL_Q7, COL_Q8, COL_Q33, COL_Q34, COL_Q50, COL_Q53]].isnull().all(axis=1)
print(f'Rows with zero text responses (kept but flagged): {all_oe_null.sum()}')
df_clean['all_oe_blank'] = all_oe_null

print(f'\nClean survey rows remaining: {len(df_clean)}')
df_clean.to_csv('survey_cleaned.csv', index=False, encoding='utf-8-sig')
print('  Saved -> survey_cleaned.csv')

# Replace df with the cleaned version for all downstream steps
df = df_clean
print('\ndf now points to the cleaned dataset.')

Dropped fully empty rows   : 0
Dropped duplicate rows     : 0
Role filled as Unknown     : 0
Dept filled as Unknown     : 0
Placeholder cells -> NaN   : 168
Rows with zero text responses (kept but flagged): 11

Clean survey rows remaining: 600
  Saved -> survey_cleaned.csv

df now points to the cleaned dataset.


In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOCK 3 — INTERVIEW TRANSCRIPT (Minutes_1_.docx)
# ══════════════════════════════════════════════════════════════════════════════
print('=' * 60)
print('INTERVIEW TRANSCRIPT — PRE-CLEANING AUDIT')
print('=' * 60)

print(f'Total paragraphs loaded    : {len(INTERVIEW_PARAS)}')

# Audit
very_short   = [p for p in INTERVIEW_PARAS if len(p) < 10]
duplicates   = len(INTERVIEW_PARAS) - len(set(INTERVIEW_PARAS))
print(f'Very short (<10 chars)     : {len(very_short)}')
print(f'Duplicate paragraphs       : {duplicates}')
if very_short:
    print('  Examples:', very_short[:5])

# Clean
seen = set()
clean_paras = []
for p in INTERVIEW_PARAS:
    # Normalise whitespace
    p_clean = re.sub(r'\s+', ' ', p).strip()
    # Drop very short or duplicate lines
    if len(p_clean) >= 10 and p_clean not in seen:
        seen.add(p_clean)
        clean_paras.append(p_clean)

removed = len(INTERVIEW_PARAS) - len(clean_paras)
print(f'\nParagraphs removed (short/duplicate): {removed}')
print(f'Clean paragraphs remaining : {len(clean_paras)}')

# Replace globals used by later steps
INTERVIEW_PARAS = clean_paras
INTERVIEW_TEXT  = '\n'.join(INTERVIEW_PARAS)
save_txt('interview_transcript.txt', INTERVIEW_TEXT)
print('interview_transcript.txt updated with cleaned version.')

INTERVIEW TRANSCRIPT — PRE-CLEANING AUDIT
Total paragraphs loaded    : 219
Very short (<10 chars)     : 35
Duplicate paragraphs       : 24
  Examples: ['OLA', 'ULRC', 'RDE', 'Finance', 'UAGC']

Paragraphs removed (short/duplicate): 36
Clean paragraphs remaining : 183
  Saved -> interview_transcript.txt
interview_transcript.txt updated with cleaned version.


In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOCK 4 — NVIVO RESULTS (results3.xlsx)
# ══════════════════════════════════════════════════════════════════════════════
print('=' * 60)
print('NVIVO RESULTS — PRE-CLEANING AUDIT')
print('=' * 60)

print(f'Total rows loaded          : {len(df_nvivo)}')
print(f'Null values in quote col   : {df_nvivo[QUOTE_COL].isnull().sum()}')
print(f'Duplicate quote rows       : {df_nvivo.duplicated(subset=[QUOTE_COL]).sum()}')

# Clean
df_nvivo_clean = df_nvivo.copy()

# Drop null quotes
before = len(df_nvivo_clean)
df_nvivo_clean = df_nvivo_clean.dropna(subset=[QUOTE_COL])
print(f'\nDropped null quote rows    : {before - len(df_nvivo_clean)}')

# Drop duplicate quotes
before = len(df_nvivo_clean)
df_nvivo_clean = df_nvivo_clean.drop_duplicates(subset=[QUOTE_COL])
print(f'Dropped duplicate quotes   : {before - len(df_nvivo_clean)}')

# Strip whitespace and collapse spaces in quote text
df_nvivo_clean[QUOTE_COL] = (df_nvivo_clean[QUOTE_COL]
                              .astype(str)
                              .str.strip()
                              .str.replace(r'\s+', ' ', regex=True))

# Drop placeholders
mask = df_nvivo_clean[QUOTE_COL].str.match(PLACEHOLDER_PATTERN, na=False)
print(f'Dropped placeholder quotes : {mask.sum()}')
df_nvivo_clean = df_nvivo_clean[~mask]

print(f'\nClean NVivo rows remaining : {len(df_nvivo_clean)}')
df_nvivo_clean.to_csv('nvivo_cleaned.csv', index=False, encoding='utf-8-sig')
print('  Saved -> nvivo_cleaned.csv')

# Rebuild NVIVO_TEXT from clean data
NVIVO_TEXT = ' '.join(df_nvivo_clean[QUOTE_COL].tolist())
save_txt('nvivo_quotes.txt', NVIVO_TEXT)
df_nvivo = df_nvivo_clean
print('nvivo_quotes.txt updated with cleaned version.')

NVIVO RESULTS — PRE-CLEANING AUDIT
Total rows loaded          : 16
Null values in quote col   : 1
Duplicate quote rows       : 0

Dropped null quote rows    : 1
Dropped duplicate quotes   : 0
Dropped placeholder quotes : 0

Clean NVivo rows remaining : 15
  Saved -> nvivo_cleaned.csv
  Saved -> nvivo_quotes.txt
nvivo_quotes.txt updated with cleaned version.


In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOCK 5 — CLEANING SUMMARY REPORT
# ══════════════════════════════════════════════════════════════════════════════
summary = [
    '=' * 60,
    'DATA CLEANING SUMMARY',
    '=' * 60,
    f'Survey  : {total_rows} rows loaded -> {len(df)} rows after cleaning',
    f'          Fully empty rows dropped, duplicates removed,',
    f'          Role/Dept blanks filled as Unknown,',
    f'          {cleaned_cells} placeholder cells converted to NaN.',
    f'Interview: {len(INTERVIEW_PARAS)} clean paragraphs (short/duplicate lines removed).',
    f'NVivo   : {len(df_nvivo)} clean rows (nulls, duplicates, placeholders removed).',
    '',
    'All downstream steps use the cleaned versions.',
    '=' * 60,
]
print('\n'.join(summary))
save_txt('cleaning_report.txt', '\n'.join(summary))

DATA CLEANING SUMMARY
Survey  : 600 rows loaded -> 600 rows after cleaning
          Fully empty rows dropped, duplicates removed,
          Role/Dept blanks filled as Unknown,
          168 placeholder cells converted to NaN.
Interview: 183 clean paragraphs (short/duplicate lines removed).
NVivo   : 15 clean rows (nulls, duplicates, placeholders removed).

All downstream steps use the cleaned versions.
  Saved -> cleaning_report.txt


---
## Part 1 — Thematic Analysis
### Step 1 (Prompt 1A) — Extract Open-Ended Survey Responses

In [16]:
records  = []
by_role  = defaultdict(list)

for _, row in df.iterrows():
    role = clean(row[COL_ROLE])
    dept = clean(row[COL_DEPT])
    for q_label, col in OPEN_COLS.items():
        val = clean(row[col])
        if not is_blank(val):
            records.append({'Role': role, 'Department': dept, 'Question': q_label, 'Response': val})
            by_role[role].append((dept, q_label, val))

df_responses = pd.DataFrame(records)
df_responses.to_csv('output_1A_responses.csv', index=False, encoding='utf-8-sig')

# Grouped readable text
lines = []
for role in sorted(by_role):
    lines.append(f'\n{"="*60}')
    lines.append(f'ROLE: {role}  ({len(by_role[role])} responses)')
    lines.append('='*60)
    for dept, q, resp in by_role[role]:
        lines.append(f'  [{dept}] {q}')
        lines.append(f'  -> {resp}\n')

save_txt('output_1A_responses.txt', '\n'.join(lines))
print(f'\nTotal open-ended responses extracted: {len(records)}')
print('Breakdown by role:')
for role, items in sorted(by_role.items()):
    print(f'  {role}: {len(items)}')

  Saved -> output_1A_responses.txt

Total open-ended responses extracted: 2585
Breakdown by role:
  Administrator: 32
  Faculty: 247
  Staff: 435
  Student: 1871


### Step 2 (Prompt 1B) — First-Level Coding of Survey Responses

In [17]:
# Each rule: (code_label, [keywords that trigger it])
# A single response can match multiple codes.
# Add or edit rules to fit your actual data.

CODING_RULES = [
    ('fear of being hacked',
        ['hack', 'hacked', 'hacking', 'intrusion']),
    ('awareness of phishing only',
        ['phishing', 'phish', 'fake email', 'suspicious link']),
    ('awareness of malware/virus',
        ['malware', 'virus', 'ransomware', 'spyware']),
    ('fear of data leak/exposure',
        ['leak', 'exposed', 'data breach', 'personal data', 'privacy']),
    ('surface-level threat awareness',
        ['scam', 'fake', 'spam', 'suspicious']),
    ('trust in institution without knowledge',
        ['trust the university', 'believe usep', 'rely on usep',
         'confident in usep', 'university handles', 'they handle']),
    ('passive confidence',
        ['safe enough', 'feel safe', 'okay naman',
         'not worried', 'no worries', 'satisfied']),
    ('unsure about institutional protection',
        ['not sure', 'unsure', "don't know", 'do not know',
         'unclear', 'hindi ko alam', 'di ko alam']),
    ('request for more training',
        ['training', 'seminar', 'workshop', 'orientation',
         'educate', 'awareness program', 'teach us']),
    ('request for stronger policies',
        ['policy', 'policies', 'guidelines', 'rules',
         'regulations', 'protocol']),
    ('request for better tools/systems',
        ['antivirus', 'vpn', 'firewall', 'system', 'software',
         'tool', 'technology', 'equipment']),
    ('weak password practices mentioned',
        ['password', 'weak password', 'change password', '2fa', 'two-factor']),
    ('concern about unauthorized access',
        ['unauthorized', 'account takeover', 'someone accessing', 'others accessing']),
    ('institutional inadequacy perceived',
        ['not enough', 'kulang', 'inadequate', 'insufficient',
         'poor', 'lacking', 'walang']),
    ('emotional overwhelm/confusion',
        ['overwhelm', 'confus', 'scared', 'fear', 'worry',
         'anxious', 'helpless', 'unsafe', 'nervous']),
    ('general positive feedback',
        ['good', 'effective', 'adequate', 'sufficient', 'improved', 'okay']),
    ('concern about identity theft',
        ['identity theft', 'identity', 'credentials stolen', 'personal info stolen']),
    ('lack of awareness of own practices',
        ["i don't know what to do", 'no idea', 'not aware', 'unaware']),
    ('awareness of backup/recovery need',
        ['backup', 'recovery', 'restore', 'data loss']),
    ('request for helpdesk/support',
        ['helpdesk', 'help desk', 'it support', 'report to', 'where to report']),
]

def apply_codes(text):
    text_lower = text.lower()
    matched = [code for code, keywords in CODING_RULES
               if any(kw.lower() in text_lower for kw in keywords)]
    return matched if matched else ['[no match - manual review needed]']

df_responses['Codes']     = df_responses['Response'].apply(apply_codes)
df_responses['Codes_str'] = df_responses['Codes'].apply(lambda c: ' | '.join(c))
df_responses.to_csv('output_1B_coded_survey.csv', index=False, encoding='utf-8-sig')

# Frequency table
code_role_counts = defaultdict(lambda: defaultdict(int))
for _, row in df_responses.iterrows():
    for code in row['Codes']:
        code_role_counts[code][row['Role']] += 1

roles_found = df_responses['Role'].unique().tolist()
freq_rows = []
for code, role_dict in code_role_counts.items():
    r = {'Code': code, 'Total': sum(role_dict.values())}
    for role in roles_found:
        r[role] = role_dict.get(role, 0)
    freq_rows.append(r)

df_freq_1B = pd.DataFrame(freq_rows).sort_values('Total', ascending=False)
df_freq_1B.to_csv('output_1B_code_frequency.csv', index=False, encoding='utf-8-sig')

print('Survey code frequency table (top 20):')
print(df_freq_1B.head(20).to_string(index=False))
print(f'\nUnique codes applied: {len(df_freq_1B)}')

Survey code frequency table (top 20):
                                 Code  Total  Student  Staff  Faculty  Administrator
    [no match - manual review needed]    838      624    109       96              9
           fear of data leak/exposure    742      546    146       43              7
            general positive feedback    508      372     84       48              4
     request for better tools/systems    242      141     72       24              5
        request for stronger policies    165       92     53       18              2
            request for more training    153       97     39       14              3
                 fear of being hacked    123       92     21        9              1
    weak password practices mentioned     88       67     12        5              4
           awareness of malware/virus     82       57     14       10              1
    concern about unauthorized access     68       37     22        7              2
        emotional overwhelm

### Step 3 (Prompt 1C) — First-Level Coding of Interview Transcript

In [18]:
# Edit OFFICES to match exactly how speaker labels appear in your Minutes_1_.docx
OFFICES = [
    'SDMD', 'CIC', 'Finance', 'HRMD', 'OUR', 'RDE',
    'Procurement', 'UAGC', 'OSA', 'OVCAA', 'OVCRE',
    'Registrar', 'Library', 'Security', 'Admin', 'IT',
]

def detect_office(line):
    for office in OFFICES:
        if re.match(rf'^{re.escape(office)}\b', line, re.IGNORECASE):
            return office
        if re.search(rf'\b{re.escape(office)}\b', line[:50], re.IGNORECASE):
            return office
    return None

current_office = 'Unknown'
interview_coded = []

for para in INTERVIEW_PARAS:
    detected = detect_office(para)
    if detected:
        current_office = detected
    if len(para) > 30:
        codes = apply_codes(para)
        interview_coded.append({
            'Office':  current_office,
            'Passage': para,
            'Codes':   ' | '.join(codes),
        })

df_interview = pd.DataFrame(interview_coded)
df_interview.to_csv('output_1C_coded_interview.csv', index=False, encoding='utf-8-sig')

# Frequency table
int_code_office = defaultdict(lambda: defaultdict(int))
for _, row in df_interview.iterrows():
    for code in row['Codes'].split(' | '):
        int_code_office[code][row['Office']] += 1

offices_found = df_interview['Office'].unique().tolist()
int_freq_rows = []
for code, off_dict in int_code_office.items():
    r = {'Code': code, 'Total': sum(off_dict.values())}
    for o in offices_found:
        r[o] = off_dict.get(o, 0)
    int_freq_rows.append(r)

df_freq_1C = pd.DataFrame(int_freq_rows).sort_values('Total', ascending=False)
df_freq_1C.to_csv('output_1C_code_frequency.csv', index=False, encoding='utf-8-sig')

print('Interview code frequency (top 15):')
print(df_freq_1C.head(15).to_string(index=False))
print(f'\nOffices detected in transcript: {offices_found}')

Interview code frequency (top 15):
                              Code  Total  Unknown  OUR  Procurement  IT  Finance  SDMD  Security  CIC  RDE
 [no match - manual review needed]     93        5   22            2  27        4    27         2    4    0
  request for better tools/systems     23        0    5            0   7        0     9         2    0    0
        fear of data leak/exposure     17        2    2            0   2        0     8         1    1    1
 weak password practices mentioned     15        0    2            1   2        0     4         3    3    0
         request for more training     14        1    1            0   2        0     9         0    0    1
              fear of being hacked     10        0    2            0   2        1     3         0    2    0
        awareness of phishing only     10        0    0            1   3        0     4         0    2    0
    surface-level threat awareness      9        0    1            1   1        0     1         0    

### Step 4 (Prompt 1D) — Group Codes into Themes & Subthemes

In [19]:
# Edit this map to reflect your actual emerging themes.
# Each theme contains subthemes; each subtheme lists which codes belong to it.

THEME_MAP = {
    'Theme 1: Surface-Level Awareness Without Deep Literacy': {
        'Subtheme 1.1 - Recognition of basic threats only': [
            'awareness of phishing only',
            'surface-level threat awareness',
            'awareness of malware/virus',
        ],
        'Subtheme 1.2 - Lack of knowledge about own practices': [
            'lack of awareness of own practices',
            'unsure about institutional protection',
        ],
    },
    'Theme 2: Fear and Emotional Distress About Cybersecurity': {
        'Subtheme 2.1 - Personal fear and anxiety': [
            'fear of being hacked',
            'emotional overwhelm/confusion',
            'fear of data leak/exposure',
            'concern about identity theft',
        ],
        'Subtheme 2.2 - Concern about unauthorized access': [
            'concern about unauthorized access',
        ],
    },
    'Theme 3: Passive Trust in the Institution': {
        'Subtheme 3.1 - Deferred responsibility to USeP': [
            'trust in institution without knowledge',
            'passive confidence',
            'general positive feedback',
        ],
        'Subtheme 3.2 - Perceived institutional inadequacy': [
            'institutional inadequacy perceived',
        ],
    },
    'Theme 4: Demand for Education, Tools, and Support': {
        'Subtheme 4.1 - Request for training and awareness programs': [
            'request for more training',
            'request for helpdesk/support',
        ],
        'Subtheme 4.2 - Request for stronger policies and tools': [
            'request for stronger policies',
            'request for better tools/systems',
            'weak password practices mentioned',
            'awareness of backup/recovery need',
        ],
    },
}

survey_totals    = df_freq_1B.set_index('Code')['Total'].to_dict()
interview_totals = df_freq_1C.set_index('Code')['Total'].to_dict()

theme_rows   = []
report_lines = []

for theme, subthemes in THEME_MAP.items():
    report_lines.append(f'\n{"="*70}')
    report_lines.append(theme)
    report_lines.append('='*70)
    t_s = t_i = 0

    for subtheme, codes in subthemes.items():
        report_lines.append(f'\n  {subtheme}')
        st_s = st_i = 0
        for code in codes:
            s = survey_totals.get(code, 0)
            i = interview_totals.get(code, 0)
            st_s += s
            st_i += i
            report_lines.append(f"    [{code}]  Survey: {s}  Interview: {i}")
        t_s += st_s
        t_i += st_i
        report_lines.append(f'  >> Subtheme totals: Survey={st_s}  Interview={st_i}')
        theme_rows.append({
            'Theme':           theme,
            'Subtheme':        subtheme,
            'Survey_hits':     st_s,
            'Interview_hits':  st_i,
            'Combined_total':  st_s + st_i,
        })

    dominant = 'Both' if t_s > 0 and t_i > 0 else ('Survey' if t_s > 0 else 'Interview')
    report_lines.append(f'\n  THEME TOTAL: Survey={t_s}  Interview={t_i}  | Dominant source: {dominant}')

df_themes = pd.DataFrame(theme_rows)
df_themes.to_csv('output_1D_themes.csv', index=False, encoding='utf-8-sig')
save_txt('output_1D_themes.txt', '\n'.join(report_lines))

print('\n'.join(report_lines))

  Saved -> output_1D_themes.txt

Theme 1: Surface-Level Awareness Without Deep Literacy

  Subtheme 1.1 - Recognition of basic threats only
    [awareness of phishing only]  Survey: 39  Interview: 10
    [surface-level threat awareness]  Survey: 12  Interview: 9
    [awareness of malware/virus]  Survey: 82  Interview: 2
  >> Subtheme totals: Survey=133  Interview=21

  Subtheme 1.2 - Lack of knowledge about own practices
    [lack of awareness of own practices]  Survey: 34  Interview: 1
    [unsure about institutional protection]  Survey: 37  Interview: 0
  >> Subtheme totals: Survey=71  Interview=1

  THEME TOTAL: Survey=204  Interview=22  | Dominant source: Both

Theme 2: Fear and Emotional Distress About Cybersecurity

  Subtheme 2.1 - Personal fear and anxiety
    [fear of being hacked]  Survey: 123  Interview: 10
    [emotional overwhelm/confusion]  Survey: 40  Interview: 4
    [fear of data leak/exposure]  Survey: 742  Interview: 17
    [concern about identity theft]  Survey: 10 

In [20]:
# ── Print representative quotes per theme ─────────────────────────────────────
print('SAMPLE SUPPORTING QUOTES PER THEME\n')
for theme, subthemes in THEME_MAP.items():
    all_codes = [c for codes in subthemes.values() for c in codes]
    print(f'\n{theme}')
    print('-'*60)
    hits = df_responses[df_responses['Codes'].apply(
        lambda c: any(code in c for code in all_codes))]
    for _, row in hits.head(3).iterrows():
        print(f'  [Survey | {row["Role"]} | {row["Department"]}]')
        print(f'  "{row["Response"][:220]}"\n')
    int_hits = df_interview[df_interview['Codes'].apply(
        lambda c: any(code in c for code in all_codes))]
    for _, row in int_hits.head(2).iterrows():
        print(f'  [Interview | {row["Office"]}]')
        print(f'  "{row["Passage"][:220]}"\n')

SAMPLE SUPPORTING QUOTES PER THEME


Theme 1: Surface-Level Awareness Without Deep Literacy
------------------------------------------------------------
  [Survey | Student | College Of Arts And Sciences]
  "None, I have never felt the threat of any malware with usep's security against cyber odds."

  [Survey | Student | College Of Arts And Sciences]
  "I don't know."

  [Survey | Student | College Of Arts And Sciences]
  "I think the most common worry that we have as students are those common problems that we encounter in cyber security, such as phishing and stuff"

  [Interview | IT]
  "I always check email addresses carefully because of a bad experience with phishing from my previous employment."

  [Interview | IT]
  "We once experienced a case where a staff member’s computer was infected with ransomware—the files were encrypted along with the virus, and the unit was immediately disconnected from the network to prevent it from spreadi"


Theme 2: Fear and Emotional Distress About C

---
## Part 2 — Content Analysis
### Step 5 (Prompt 2A) — Word Frequency Tracking Across All Files

In [21]:
TIERS = {
    'Tier 1 - Basic Threat Terms': [
        'email', 'phishing', 'hacking', 'hacked', 'virus', 'malware',
        'scam', 'ransomware', 'suspicious link', 'fake',
    ],
    'Tier 2 - Practice Terms': [
        'password', '2fa', 'two-factor', 'antivirus', 'update',
        'backup', 'logout', 'lock', 'privacy settings', 'vpn',
    ],
    'Tier 3 - Advanced Terms': [
        'encryption', 'access control', 'incident response', 'data breach protocol',
        'vulnerability', 'firewall', 'penetration testing', 'network security',
        'authentication', 'zero trust',
    ],
    'Tier 4 - Emotional Terms': [
        'scared', 'fear', 'worried', 'overwhelmed', 'unsafe',
        'concerned', 'confused', 'helpless', 'cautious', 'confident',
    ],
    'Tier 5 - Institutional Terms': [
        'training', 'seminar', 'policy', 'guidelines', 'sdmd',
        'budget', 'helpdesk', 'orientation', 'reporting', 'framework',
    ],
}

SURVEY_FULL_TEXT = ' '.join(df_responses['Response'].tolist())
INTERVIEW_TEXT   = Path('interview_transcript.txt').read_text(encoding='utf-8')
NVIVO_TEXT       = Path('nvivo_quotes.txt').read_text(encoding='utf-8')
ALL_TEXT         = ' '.join([SURVEY_FULL_TEXT, INTERVIEW_TEXT, NVIVO_TEXT])

freq_rows = []
for tier, terms in TIERS.items():
    for term in terms:
        freq_rows.append({
            'Tier':      tier,
            'Term':      term,
            'Survey':    count_term(term, SURVEY_FULL_TEXT),
            'Interview': count_term(term, INTERVIEW_TEXT),
            'NVivo':     count_term(term, NVIVO_TEXT),
            'Total':     count_term(term, ALL_TEXT),
        })

df_wordfreq = pd.DataFrame(freq_rows)
df_wordfreq.to_csv('output_2A_word_frequency.csv', index=False, encoding='utf-8-sig')

print(df_wordfreq.to_string(index=False))
print('\n-- Tier Totals --')
print(df_wordfreq.groupby('Tier')['Total'].sum().to_string())

                        Tier                 Term  Survey  Interview  NVivo  Total
 Tier 1 - Basic Threat Terms                email      27         24      6     57
 Tier 1 - Basic Threat Terms             phishing      38          9      1     48
 Tier 1 - Basic Threat Terms              hacking      61          5      0     66
 Tier 1 - Basic Threat Terms               hacked      33          5      0     38
 Tier 1 - Basic Threat Terms                virus      13          1      0     14
 Tier 1 - Basic Threat Terms              malware      31          1      0     32
 Tier 1 - Basic Threat Terms                 scam       2          1      2      5
 Tier 1 - Basic Threat Terms           ransomware       5          1      0      6
 Tier 1 - Basic Threat Terms      suspicious link       0          1      0      1
 Tier 1 - Basic Threat Terms                 fake       4          3      1      8
     Tier 2 - Practice Terms             password      42          3      4     49
    

### Step 6 (Prompt 2B) — Comparative Analysis by Role and Office

In [22]:
# Table 1 — Tier 1 vs Tier 3 by Role (Survey)
tier1_terms = TIERS['Tier 1 - Basic Threat Terms']
tier3_terms = TIERS['Tier 3 - Advanced Terms']

role_rows = []
for role in df_responses['Role'].unique():
    role_text = ' '.join(
        df_responses[df_responses['Role'] == role]['Response'].tolist())
    t1 = sum(count_term(t, role_text) for t in tier1_terms)
    t3 = sum(count_term(t, role_text) for t in tier3_terms)
    ratio = round(t1 / t3, 1) if t3 > 0 else 'inf'
    role_rows.append({'Role': role, 'Tier1_Basic': t1, 'Tier3_Advanced': t3, 'Ratio_T1:T3': ratio})

df_role_comp = pd.DataFrame(role_rows)
df_role_comp.to_csv('output_2B_role_comparison.csv', index=False, encoding='utf-8-sig')
print('Table 1 - Tier 1 vs Tier 3 by Role:')
print(df_role_comp.to_string(index=False))

Table 1 - Tier 1 vs Tier 3 by Role:
         Role  Tier1_Basic  Tier3_Advanced  Ratio_T1:T3
      Student          143              49          2.9
        Staff           48              35          1.4
      Faculty           20               7          2.9
Administrator            3               2          1.5


In [23]:
# Table 2 — Technical vs Non-Technical Offices (Interview, Tier 3)
TECH_OFFICES     = ['SDMD', 'CIC', 'IT']
NON_TECH_OFFICES = ['Finance', 'HRMD', 'OUR', 'RDE', 'Procurement',
                    'UAGC', 'OSA', 'Registrar', 'Library', 'Admin']

def office_text(office_list):
    return ' '.join(
        df_interview[df_interview['Office'].isin(office_list)]['Passage'].tolist())

tech_text    = office_text(TECH_OFFICES)
nontech_text = office_text(NON_TECH_OFFICES)

office_comp = [{'Term': t,
                'Technical':     count_term(t, tech_text),
                'Non-Technical': count_term(t, nontech_text)}
               for t in tier3_terms]

df_office_comp = pd.DataFrame(office_comp)
df_office_comp.to_csv('output_2B_office_comparison.csv', index=False, encoding='utf-8-sig')
print('\nTable 2 - Advanced Terms: Technical vs Non-Technical Offices:')
print(df_office_comp.to_string(index=False))


Table 2 - Advanced Terms: Technical vs Non-Technical Offices:
                Term  Technical  Non-Technical
          encryption          0              0
      access control          2              0
   incident response          0              0
data breach protocol          0              0
       vulnerability          0              0
            firewall          0              0
 penetration testing          0              0
    network security          0              0
      authentication          6              1
          zero trust          0              0


In [24]:
# Table 3 — Emotional Term Count per Office (ranked)
tier4_terms = TIERS['Tier 4 - Emotional Terms']

emotion_rows = []
for office in df_interview['Office'].unique():
    txt = ' '.join(df_interview[df_interview['Office'] == office]['Passage'].tolist())
    score = sum(count_term(t, txt) for t in tier4_terms)
    emotion_rows.append({'Office': office, 'Emotional_Term_Count': score})

df_emotion = (pd.DataFrame(emotion_rows)
                .sort_values('Emotional_Term_Count', ascending=False)
                .reset_index(drop=True))
df_emotion.to_csv('output_2B_emotion_by_office.csv', index=False, encoding='utf-8-sig')
print('\nTable 3 - Emotional Distress Ranking by Office:')
print(df_emotion.to_string(index=False))


Table 3 - Emotional Distress Ranking by Office:
     Office  Emotional_Term_Count
        OUR                     3
         IT                     3
        CIC                     2
   Security                     1
Procurement                     1
       SDMD                     1
    Unknown                     0
    Finance                     0
        RDE                     0


In [25]:
# Table 4 — Top 5 most frequent (all tiers) vs Bottom 5 Tier 3
top5_all    = df_wordfreq.nlargest(5, 'Total')[['Term', 'Tier', 'Total']]
bottom5_t3  = (df_wordfreq[df_wordfreq['Tier'] == 'Tier 3 - Advanced Terms']
               .nsmallest(5, 'Total')[['Term', 'Total']])

print('Top 5 Most Frequent Terms (all sources):')
print(top5_all.to_string(index=False))
print('\nBottom 5 Tier 3 (Advanced) Terms - least mentioned:')
print(bottom5_t3.to_string(index=False))

top5_all.to_csv('output_2B_top5_terms.csv',   index=False, encoding='utf-8-sig')
bottom5_t3.to_csv('output_2B_bottom5_t3.csv', index=False, encoding='utf-8-sig')

Top 5 Most Frequent Terms (all sources):
          Term                         Tier  Total
      training Tier 5 - Institutional Terms     71
       hacking  Tier 1 - Basic Threat Terms     66
         email  Tier 1 - Basic Threat Terms     57
authentication      Tier 3 - Advanced Terms     57
      password      Tier 2 - Practice Terms     49

Bottom 5 Tier 3 (Advanced) Terms - least mentioned:
                Term  Total
data breach protocol      0
          zero trust      0
       vulnerability      1
 penetration testing      1
    network security      2


---
## Step 7 — Export Full Combined Report

In [26]:
sep = '\n' + '='*70 + '\n'

report = [
    'CyberSafe USeP - Full Analysis Report',
    'Generated by: cybersafe_analysis.ipynb (Pure Python, no AI, no API)',
    sep,
    'PART 1 - THEMATIC ANALYSIS',
    sep,
    'STEP 1A - Responses Extracted',
    f'Total: {len(df_responses)}',
    df_responses["Role"].value_counts().to_string(),
    sep,
    'STEP 1B - Survey Code Frequency (Top 20)',
    df_freq_1B.head(20).to_string(index=False),
    sep,
    'STEP 1C - Interview Code Frequency (Top 15)',
    df_freq_1C.head(15).to_string(index=False),
    sep,
    'STEP 1D - Themes and Subthemes',
    Path('output_1D_themes.txt').read_text(encoding='utf-8'),
    sep,
    'PART 2 - CONTENT ANALYSIS',
    sep,
    'STEP 2A - Word Frequency Table',
    df_wordfreq.to_string(index=False),
    '\nTier Totals:',
    df_wordfreq.groupby('Tier')['Total'].sum().to_string(),
    sep,
    'STEP 2B - Role Comparison (Tier 1 vs Tier 3)',
    df_role_comp.to_string(index=False),
    '\nTechnical vs Non-Technical Offices:',
    df_office_comp.to_string(index=False),
    '\nEmotional Distress Ranking by Office:',
    df_emotion.to_string(index=False),
    '\nTop 5 Most Frequent Terms:',
    top5_all.to_string(index=False),
    '\nBottom 5 Tier 3 Terms:',
    bottom5_t3.to_string(index=False),
]

save_txt('FULL_REPORT.txt', '\n'.join(report))

print('\nAll output files:')
for f in sorted(Path('.').glob('output_*.csv')):
    print(f'  {f.name}')
for f in sorted(Path('.').glob('output_*.txt')):
    print(f'  {f.name}')
print('  FULL_REPORT.txt')

  Saved -> FULL_REPORT.txt

All output files:
  output_1A_responses.csv
  output_1B_code_frequency.csv
  output_1B_coded_survey.csv
  output_1C_code_frequency.csv
  output_1C_coded_interview.csv
  output_1D_themes.csv
  output_2A_word_frequency.csv
  output_2B_bottom5_t3.csv
  output_2B_emotion_by_office.csv
  output_2B_office_comparison.csv
  output_2B_role_comparison.csv
  output_2B_top5_terms.csv
  output_1A_responses.txt
  output_1D_themes.txt
  FULL_REPORT.txt
